# U3 AquaVigía: ejecución integrada de punta a punta (S13)

Dimensión de **Celia Patricia Apaza Hilasaca**: temperatura_c, oxigeno_mgl. Evento de negocio: **orden de inspección** en `orden-eventos`.

Una sola ejecución que junta las dos unidades y sabe si salió bien:

| Paso | Qué hace | Si falla |
|---|---|---|
| 1. Compuertas previas | Comprueba 9 dependencias: artefactos de la U1 y la U2, Kafka, MQTT, Pushgateway, Prometheus y Grafana | Se detiene en el acto y dice qué falta |
| 2. Batch (Unidad 1) | Lee el Parquet particionado de la U1 y aplica compuertas de calidad | Se detiene si los datos no sirven |
| 3. Streaming (Unidad 2) | Lee Kafka durante una ventana fija, valida, deduplica, pronostica sin estado, calcula el riesgo y publica el evento de negocio | Se detiene si el stream no entregó filas |
| 4. KPI de tiempo | Latencia de punta a punta y duración del micro-lote con **p50 y p95**, no solo el promedio, con compuerta de plausibilidad | La compuerta usa el p50; el p95 solo advierte |
| 5. Prometheus | Publica las métricas de la corrida y confirma que Prometheus devuelve **los valores recién publicados** | Se detiene si no coinciden |
| 6. Manifiesto | Escribe `manifiesto_patricia_<runId>.json` y lo compara con la corrida anterior | |

In [1]:
import os, json, time, math, shutil, uuid, socket, platform, urllib.request, urllib.parse
from datetime import datetime, timezone
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType, MapType
from pyspark.ml import PipelineModel
from pyspark.sql.streaming import StreamingQueryListener

INTEGRANTE = "Patricia Apaza"
ID = "patricia"
VARIABLES = ['temperatura_c', 'oxigeno_mgl']          # U2: variables del sensor USGS
VARIABLES_U1 = ['temperatura_c', 'oxigeno_disuelto_mgl']       # U1: las mismas magnitudes en el dataset batch
FLAG_U1 = "fuera_de_rango"
TOPIC_ENTRADA = "agua-usgs-eventos-v2"
TOPIC_NEGOCIO = "orden-eventos"
TIPO_NEGOCIO = "orden_inspeccion"
BROKER = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS = "http://prometheus:9090"
GRAFANA = "http://grafana:3000"
MQTT = ("mosquitto", 1883)

# Rutas dentro del contenedor (/opt = pyspark/sesiones del repositorio). artifacts/ no se versiona:
# cada copia los genera ejecutando el cuaderno de la U1 y el 12b; la compuerta previa dice cual falta.
RUTA_U1_GOLD = f"/opt/AquaVigia_Producto_U1/{ID}/artifacts/mediciones_particionado"
RUTA_U1_MODELO = f"/opt/AquaVigia_Producto_U1/{ID}/artifacts/modelo_alerta_siguiente_mes"
RUTA_MODELOS = f"/opt/AquaVigia_Producto_U2/{ID}/artifacts/modelos_{ID}"
DIR_U3 = f"/opt/AquaVigia_Producto_U3/{ID}/artifacts"
DIR_MANIFIESTOS = f"{DIR_U3}/manifiestos"
CHK = f"{DIR_U3}/chk_integracion"

FILAS_U1_ESPERADAS = 1245817   # filas verificadas por el cuaderno 04_producto_u1 (ida y vuelta sin perdida)
DURACION_S = 240               # ventana fija del streaming: una ejecucion integrada tiene que terminar
TRIGGER_S = 10
MAX_POR_LOTE = 600
RIESGO_ALERTA = 0.5
ENFRIAMIENTO_MIN = 60
WATERMARK = "85 minutes"       # medido en el 12a: p99 del retraso del USGS x 1.2
P50_LATENCIA_MAX_S = 60        # plausibilidad: disparador de 10 s + MQTT + puente; mas de 1 min tipico = pipeline atascado
P95_LATENCIA_AVISO_S = 300     # advertencia: la cola peor que 5 min indica lag acumulado
DIRECCION = {"temperatura_c": "alta", "oxigeno_mgl": "baja", "ph": "ambas", "turbidez_fnu": "alta"}

spark = (SparkSession.builder.appName(f"u3-integracion-{ID}").master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "4").config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INICIO = time.time()
manifiesto = {"runId": RUN_ID, "integrante": INTEGRANTE, "inicioUtc": datetime.now(timezone.utc).isoformat(),
              "versiones": {"spark": spark.version, "pyspark": pyspark.__version__, "python": platform.python_version()},
              "parametros": {"duracionS": DURACION_S, "triggerS": TRIGGER_S, "maxPorLote": MAX_POR_LOTE,
                             "riesgoAlerta": RIESGO_ALERTA, "watermark": WATERMARK,
                             "p50LatenciaMaxS": P50_LATENCIA_MAX_S, "p95LatenciaAvisoS": P95_LATENCIA_AVISO_S},
              "compuertasPrevias": {}, "batch": {}, "streaming": {}, "kpiTiempo": {}, "observabilidad": {},
              "advertencias": []}
print("runId:", RUN_ID, "| Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-330b8866-b4e2-4d97-8b74-9044b4217150;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

runId: 20261113T163113Z | Spark 4.2.0


## 1. Compuertas previas

Antes de procesar nada se comprueba que cada dependencia existe y responde. Si una falla, la ejecución se detiene en el primer segundo con el nombre de lo que falta, en vez de fallar minutos después dentro de Spark con un error sobre otra cosa.

In [2]:
def http_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def tcp_ok(host, puerto):
    try:
        socket.create_connection((host, puerto), timeout=5).close(); return True
    except OSError:
        return False

def offsets_topico(topic):
    """Suma de los ultimos offsets del topico (mensajes guardados). None si el topico no existe o Kafka no responde."""
    try:
        df = (spark.read.format("kafka").option("kafka.bootstrap.servers", BROKER).option("subscribe", topic)
              .option("startingOffsets", "earliest").option("endingOffsets", "latest").load())
        return df.count()
    except Exception:
        return None

def existe_spark_dir(ruta):
    return os.path.isdir(ruta) and os.path.exists(os.path.join(ruta, "_SUCCESS"))

mensajes_entrada = offsets_topico(TOPIC_ENTRADA)
mensajes_negocio_antes = offsets_topico(TOPIC_NEGOCIO)
compuertas = {
    "Parquet de la U1 (gold) existe": existe_spark_dir(RUTA_U1_GOLD),
    "Modelo de la U1 existe": os.path.isdir(os.path.join(RUTA_U1_MODELO, "metadata")),
    "Modelos de la U2 (12b) existen": os.path.exists(f"{RUTA_MODELOS}/meta.json")
                                     and all(os.path.isdir(f"{RUTA_MODELOS}/modelo_{v}_1h") for v in VARIABLES),
    "Kafka: topico " + TOPIC_ENTRADA + " con datos": bool(mensajes_entrada),
    "Kafka: topico " + TOPIC_NEGOCIO + " existe": mensajes_negocio_antes is not None,
    "MQTT (mosquitto:1883) acepta conexiones": tcp_ok(*MQTT),
    "Pushgateway listo": http_ok(PUSHGATEWAY + "/-/ready"),
    "Prometheus listo": http_ok(PROMETHEUS + "/-/ready"),
    "Grafana saludable": http_ok(GRAFANA + "/api/health"),
}
for nombre, ok in compuertas.items():
    print(("OK    " if ok else "FALLA ") + nombre)
manifiesto["compuertasPrevias"] = compuertas
manifiesto["kafka"] = {"mensajesEntrada": mensajes_entrada, "mensajesNegocioAntes": mensajes_negocio_antes}
fallidas = [n for n, ok in compuertas.items() if not ok]
assert not fallidas, "Compuertas que fallan: " + str(fallidas)

OK    Parquet de la U1 (gold) existe
OK    Modelo de la U1 existe
OK    Modelos de la U2 (12b) existen
OK    Kafka: topico agua-usgs-eventos-v2 con datos
OK    Kafka: topico orden-eventos existe
OK    MQTT (mosquitto:1883) acepta conexiones
OK    Pushgateway listo
OK    Prometheus listo
OK    Grafana saludable


## 2. Batch: Parquet de la Unidad 1 con compuertas de calidad

El Parquet *gold* que produjo `04_producto_u1_patricia` (particionado por `region`). Cuatro tipos de compuerta (S13, Tabla 3):

| Tipo | Compuerta |
|---|---|
| Existencia | Hay filas |
| Completitud | Están las 1 245 817 filas verificadas en la U1; las columnas que se usan no tienen nulos |
| Rango | Los valores caen en el rango **físico** del sensor |
| Plausibilidad | La mediana (p50) de cada variable cae en un rango creíble para agua superficial |

In [3]:
RANGO_FISICO_U1 = {"temperatura_c": (-5, 45), "oxigeno_disuelto_mgl": (0, 25), "ph": (0, 14), "turbidez_ntu": (0, 4000)}
# Plausibilidad: mediana creible para rios y lagos (no es el rango fisico del sensor)
P50_PLAUSIBLE_U1 = {"temperatura_c": (2, 30), "oxigeno_disuelto_mgl": (3, 14), "ph": (5.5, 9.0), "turbidez_ntu": (0, 100)}

gold = spark.read.parquet(RUTA_U1_GOLD)
total = gold.count()
regiones = sorted(r[0] for r in gold.select("region").distinct().collect())
agg = []
for v in VARIABLES_U1:
    lo, hi = RANGO_FISICO_U1[v]
    agg += [F.sum(F.col(v).isNull().cast("int")).alias(f"{v}__nulos"),
            F.sum(((F.col(v) < lo) | (F.col(v) > hi)).cast("int")).alias(f"{v}__fuera_fisico"),
            F.expr(f"percentile_approx({v}, 0.5)").alias(f"{v}__p50"),
            F.expr(f"percentile_approx({v}, 0.95)").alias(f"{v}__p95"),
            F.avg(v).alias(f"{v}__promedio")]
k = gold.agg(*agg).first().asDict()

compuertas_calidad = {"Hay filas": total > 0,
                      f"Completitud: {FILAS_U1_ESPERADAS:,} filas de la U1".replace(",", " "): total == FILAS_U1_ESPERADAS}
for v in VARIABLES_U1:
    compuertas_calidad[f"{v} sin nulos"] = (k[f"{v}__nulos"] or 0) == 0
    compuertas_calidad[f"{v} dentro del rango fisico"] = (k[f"{v}__fuera_fisico"] or 0) == 0
    lo, hi = P50_PLAUSIBLE_U1[v]
    compuertas_calidad[f"{v}: p50 plausible ({lo} a {hi})"] = lo <= k[f"{v}__p50"] <= hi
for nombre, ok in compuertas_calidad.items():
    print(("OK    " if ok else "FALLA ") + nombre)

print(f"\nFilas: {total:,} | regiones ({len(regiones)}): {', '.join(regiones)}")
for v in VARIABLES_U1:
    print(f"  {v:<22} p50={k[v + '__p50']:.2f}  p95={k[v + '__p95']:.2f}  promedio={k[v + '__promedio']:.2f}")
print(f"\nIndicador de la U1 ({FLAG_U1}):")
dist = gold.groupBy(FLAG_U1).count().orderBy(FLAG_U1).toPandas()
dist["pct"] = (100 * dist["count"] / total).round(2)
print(dist.to_string(index=False))

manifiesto["batch"] = {"filas": total, "regiones": regiones, "kpi": {kk: (float(vv) if vv is not None else None) for kk, vv in k.items()},
                       "indicadorU1": dict(zip(dist[FLAG_U1].astype(str), dist["count"].astype(int).tolist())),
                       "calidad": compuertas_calidad}
assert all(compuertas_calidad.values()), "La calidad del batch no pasa: " + str([n for n, ok in compuertas_calidad.items() if not ok])

OK    Hay filas
OK    Completitud: 1 245 817 filas de la U1
OK    temperatura_c sin nulos
OK    temperatura_c dentro del rango fisico
OK    temperatura_c: p50 plausible (2 a 30)
OK    oxigeno_disuelto_mgl sin nulos
OK    oxigeno_disuelto_mgl dentro del rango fisico
OK    oxigeno_disuelto_mgl: p50 plausible (3 a 14)

Filas: 1,245,817 | regiones (8): Arequipa, Cusco, Junín, Lambayeque, Lima, Loreto, Piura, Puno
  temperatura_c          p50=16.00  p95=25.00  promedio=16.01
  oxigeno_disuelto_mgl   p50=7.00  p95=9.63  promedio=7.00

Indicador de la U1 (fuera_de_rango):
 fuera_de_rango  count   pct
              0 964746 77.44
              1 281071 22.56


## 3. Streaming: el pipeline de la Unidad 2 en una ventana fija

Lo mismo que el 12c v3, pero acotado en el tiempo (`DURACION_S`): una ejecución integrada tiene que **terminar** para poder comprobarse y registrarse. Lee solo lo nuevo (`latest`), así que el productor tiene que estar enviando.

Cada micro-lote: validación de esquema y rango físico → deduplicación dentro del watermark → pronóstico a 1 hora **sin estado** (el contexto viaja en el evento) → riesgo → evento de negocio con `evento_id` idempotente → error en vivo frente a la persistencia.

In [4]:
VARS_CONOCIDAS = ["ph", "oxigeno_mgl", "turbidez_fnu", "conductividad_uscm", "temperatura_c", "nitrato_mgl"]
RANGO_FISICO = {"temperatura_c": (-5, 45), "oxigeno_mgl": (0, 25), "ph": (0, 14),
                "turbidez_fnu": (0, 4000), "conductividad_uscm": (0, 100000), "nitrato_mgl": (0, 100)}
esquema_v2 = (StructType()
    .add("version_esquema", IntegerType())
    .add("evento_id", StringType()).add("estacion_id", StringType()).add("variable", StringType())
    .add("codigo_parametro", StringType()).add("valor", DoubleType())
    .add("fecha_medicion_utc", StringType()).add("fecha_ingesta_utc", StringType())
    .add("retraso_min", DoubleType()).add("origen", StringType()).add("fuente", StringType())
    .add("contexto", StructType()
         .add("rezagos", MapType(StringType(), DoubleType()))
         .add("simultaneas", MapType(StringType(), DoubleType()))))

META = json.load(open(f"{RUTA_MODELOS}/meta.json", encoding="utf-8"))
MODELOS = {v: PipelineModel.load(f"{RUTA_MODELOS}/modelo_{v}_1h") for v in VARIABLES}
print("Modelos del 12b cargados (aqui no se entrena nada):", list(MODELOS))

def parsear(df):
    rango = F.lit(False)
    for v, (lo, hi) in RANGO_FISICO.items():
        rango = F.when(F.col("variable") == v, (F.col("valor") >= lo) & (F.col("valor") <= hi)).otherwise(rango)
    d = (df.select(F.col("timestamp").alias("kafka_ts"), F.col("value").cast("string").alias("json"))
           .withColumn("e", F.from_json("json", esquema_v2)).select("kafka_ts", "e.*")
           .withColumn("ts", F.to_timestamp("fecha_medicion_utc")))
    ok = (F.col("estacion_id").isNotNull() & F.col("variable").isNotNull() & F.col("valor").isNotNull() & F.col("ts").isNotNull())
    return d.withColumn("validacion", F.when(~ok, "esquema_invalido")
                        .when(~F.col("variable").isin(VARS_CONOCIDAS), "variable_desconocida")
                        .when(~rango, "fuera_de_rango_fisico").otherwise("valido"))

def columna_feature(f, v):
    if f == v: return F.col("valor")
    if f.startswith(f"{v}_l"): return F.col("contexto.rezagos").getItem(f.split("_")[-1])
    if f == "delta_1h": return F.col("valor") - F.col("contexto.rezagos").getItem("l12")
    if f == "hora_sin": return F.sin((F.col("ts5") % 86400) / 3600.0 * 2 * math.pi / 24)
    if f == "hora_cos": return F.cos((F.col("ts5") % 86400) / 3600.0 * 2 * math.pi / 24)
    return F.col("contexto.simultaneas").getItem(f)

def inferir(d):
    """Sin estado: cada fila se pronostica sola con su contexto."""
    salidas = []
    for v, modelo in MODELOS.items():
        feats = META["variables"][v]["features"]
        x = (d.where(F.col("variable") == v)
              .withColumn("ts5", (F.floor(F.unix_timestamp("ts") / 300) * 300).cast("long"))
              .withColumn("ultimo", F.col("valor")))
        x = x.select("evento_id", "estacion_id", "variable", "ts", "ts5", "ultimo", "retraso_min", "origen", "kafka_ts",
                     *[columna_feature(f, v).alias(f) for f in feats if f != v], F.col("valor").alias(v)).dropna(subset=feats)
        salidas.append(modelo.transform(x).select("evento_id", "estacion_id", "variable", "ts5", "ultimo",
                                                  F.col("prediction").alias("pred"), "retraso_min", "origen",
                                                  F.col("kafka_ts").cast("double").alias("kafka_epoch")))
    out = salidas[0]
    for s in salidas[1:]: out = out.unionByName(s)
    return out

def ncdf(z): return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def riesgo(est, v, pred):
    u = META["umbrales"].get(f"{est}|{v}"); s = META["variables"][v]["sigma"]
    if not u or s <= 0: return None, None
    d = DIRECCION.get(v, "ambas")
    p_alta = 1 - ncdf((u["p95"] - pred) / s); p_baja = ncdf((u["p05"] - pred) / s)
    return (p_alta if d == "alta" else p_baja if d == "baja" else min(1.0, p_alta + p_baja)), u

def evento_negocio(r, p, u):
    return {"version_esquema": 1,
            "evento_id": str(uuid.uuid5(uuid.NAMESPACE_URL, f"{TIPO_NEGOCIO}|{r['estacion_id']}|{r['variable']}|{int(r['ts5'])}")),
            "tipo": TIPO_NEGOCIO, "integrante": INTEGRANTE, "estacion_id": r["estacion_id"], "variable": r["variable"],
            "valor_actual": round(float(r["ultimo"]), 3), "pronostico_1h": round(float(r["pred"]), 3), "riesgo": round(p, 3),
            "umbral_p05": u["p05"], "umbral_p95": u["p95"], "direccion": DIRECCION.get(r["variable"]),
            "ts_medicion_utc": datetime.fromtimestamp(int(r["ts5"]), timezone.utc).isoformat(),
            "creado_utc": datetime.now(timezone.utc).isoformat(), "estado": "creada", "fuente": "USGS",
            "modelo": META["variables"][r["variable"]]["ganador"], "evento_origen_id": r["evento_id"], "runId": RUN_ID}

def publicar_negocio(eventos):
    if eventos:
        (spark.createDataFrame([(f"{e['estacion_id']}|{e['variable']}", json.dumps(e, ensure_ascii=False)) for e in eventos], ["key", "value"])
         .write.format("kafka").option("kafka.bootstrap.servers", BROKER).option("topic", TOPIC_NEGOCIO).save())

Modelos del 12b cargados (aqui no se entrena nada): ['temperatura_c', 'oxigeno_mgl']


In [5]:
PEND, ULT_ALERTA = {}, {}
ERR = {v: {m: {"n": 0, "abs": 0.0, "sq": 0.0} for m in ("modelo", "persistencia")} for v in VARIABLES}
TOT = {"lotes_con_datos": 0, "pronosticos": 0, "eventos_negocio": 0, "riesgo_max": 0.0}
LAT_E2E_S, RETRASO_USGS_MIN, DURACION_LOTE_MS, MICROLOTES = [], [], [], []

def medir_error(pdf):
    for r in pdf.itertuples():
        PEND[(r.estacion_id, r.variable, int(r.ts5) + 3600)] = (float(r.pred), float(r.ultimo))
    for r in pdf.itertuples():
        k = (r.estacion_id, r.variable, int(r.ts5))
        if k in PEND:
            pred, ult = PEND.pop(k); real = float(r.ultimo)
            for m, e in (("modelo", real - pred), ("persistencia", real - ult)):
                a = ERR[r.variable][m]; a["n"] += 1; a["abs"] += abs(e); a["sq"] += e * e

def lote(df, bid):
    pdf = inferir(df).toPandas()
    ahora = time.time()
    MICROLOTES.append({"batchId": int(bid), "pronosticos": int(len(pdf))})
    if pdf.empty: return
    TOT["lotes_con_datos"] += 1; TOT["pronosticos"] += len(pdf)
    # Latencia de punta a punta: desde que el puente dejo el evento en Kafka hasta que Spark lo pronostico
    # (el timestamp de Kafka se pasa a segundos epoch dentro de Spark: no depende de la zona horaria)
    LAT_E2E_S.extend((ahora - pdf["kafka_epoch"]).tolist())
    RETRASO_USGS_MIN.extend(pdf.loc[pdf["origen"] == "vivo", "retraso_min"].dropna().tolist())
    eventos = []
    for r in pdf.sort_values("ts5").to_dict("records"):
        p, u = riesgo(r["estacion_id"], r["variable"], r["pred"])
        if p is None: continue
        TOT["riesgo_max"] = max(TOT["riesgo_max"], p)
        k = (r["estacion_id"], r["variable"])
        if p >= RIESGO_ALERTA and r["ts5"] - ULT_ALERTA.get(k, 0) >= ENFRIAMIENTO_MIN * 60:
            eventos.append(evento_negocio(r, p, u)); ULT_ALERTA[k] = r["ts5"]
    publicar_negocio(eventos); TOT["eventos_negocio"] += len(eventos)
    medir_error(pdf)

class Observador(StreamingQueryListener):
    def onQueryStarted(self, e): pass
    def onQueryIdle(self, e): pass
    def onQueryTerminated(self, e): pass
    def onQueryProgress(self, e):
        p = e.progress
        if p.name == f"u3_{ID}" and p.numInputRows > 0:
            DURACION_LOTE_MS.append(p.durationMs.get("triggerExecution", 0))

for q in spark.streams.active:
    if q.name == f"u3_{ID}": q.stop()
if "OBS_U3" not in globals():
    OBS_U3 = Observador(); spark.streams.addListener(OBS_U3)
if os.path.exists(CHK): shutil.rmtree(CHK)
os.makedirs(DIR_MANIFIESTOS, exist_ok=True)

entrada = parsear(spark.readStream.format("kafka").option("kafka.bootstrap.servers", BROKER)
                  .option("subscribe", TOPIC_ENTRADA).option("startingOffsets", "latest")
                  .option("maxOffsetsPerTrigger", MAX_POR_LOTE).option("failOnDataLoss", "false").load())
validos = entrada.where((F.col("validacion") == "valido") & F.col("variable").isin(VARIABLES))
sin_dup = validos.withWatermark("ts", WATERMARK).dropDuplicatesWithinWatermark(["estacion_id", "variable", "ts"])
q = (sin_dup.writeStream.queryName(f"u3_{ID}").foreachBatch(lote).option("checkpointLocation", CHK)
     .trigger(processingTime=f"{TRIGGER_S} seconds").start())
print(f"Streaming {DURACION_S} s ... (el productor debe estar enviando)")
q.awaitTermination(DURACION_S)
# Parada ordenada: si hay un micro-lote en curso se espera a que termine (hasta 30 s).
# Detenerlo a la mitad cancela ese lote y Spark imprime un error largo al cerrar (bitacora, fila 4).
limite_parada = time.time() + 30
while q.isActive and q.status["isTriggerActive"] and time.time() < limite_parada:
    time.sleep(0.2)
q.stop()
print(f"Micro-lotes: {len(MICROLOTES)} | con datos: {TOT['lotes_con_datos']} | pronosticos: {TOT['pronosticos']} | "
      f"eventos de negocio: {TOT['eventos_negocio']} | riesgo maximo: {TOT['riesgo_max']:.3f}")

Streaming 240 s ... (el productor debe estar enviando)


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

Micro-lotes: 23 | con datos: 22 | pronosticos: 1015 | eventos de negocio: 4 | riesgo maximo: 0.745


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:404: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true, but has reached the error below and can not continue. Note that 'spark.sql.execution.arrow.pyspark.fallback.enabled' does not have an effect on failures in the middle of computation.
  An error occurred while calling o11400.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:126)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:122)
	at java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAc

### 3.1 Error en producción y compuertas de salida

Una ejecución que termina sin error no demuestra que produjo algo útil (caso GitLab, S13). Por eso se comprueba el **resultado**: que el stream entregó filas y que los eventos de negocio emitidos están de verdad en Kafka.

In [6]:
filas_err = []
for v in VARIABLES:
    fila = {"variable": v, "RMSE_validacion_12b": round(META["variables"][v]["rmse"], 4),
            "RMSE_validacion_persistencia": round(META["variables"][v]["rmse_persistencia"], 4)}
    for m in ("modelo", "persistencia"):
        a = ERR[v][m]
        fila[f"RMSE_vivo_{m}"] = round(math.sqrt(a["sq"] / a["n"]), 4) if a["n"] else None
        fila[f"MAE_vivo_{m}"] = round(a["abs"] / a["n"], 4) if a["n"] else None
    fila["comparaciones"] = ERR[v]["modelo"]["n"]
    filas_err.append(fila)
err = pd.DataFrame(filas_err)
print(err.to_string(index=False))

mensajes_negocio_despues = offsets_topico(TOPIC_NEGOCIO)
publicados = (mensajes_negocio_despues or 0) - (mensajes_negocio_antes or 0)
compuertas_salida = {
    "El stream entrego filas": TOT["pronosticos"] > 0,
    f"Los {TOT['eventos_negocio']} eventos de negocio emitidos estan en Kafka": publicados == TOT["eventos_negocio"],
}
for nombre, ok in compuertas_salida.items():
    print(("OK    " if ok else "FALLA ") + nombre)
for r in filas_err:
    if r["comparaciones"] == 0:
        manifiesto["advertencias"].append(f"{r['variable']}: sin comparaciones en vivo (la ventana no cubrio 1 h de tiempo de evento)")
    elif r["RMSE_vivo_modelo"] > r["RMSE_vivo_persistencia"]:
        manifiesto["advertencias"].append(f"{r['variable']}: en esta corrida la persistencia gano al modelo")
for a in manifiesto["advertencias"]:
    print("ADVERTENCIA:", a)
manifiesto["streaming"] = {"duracionS": DURACION_S, "microlotes": len(MICROLOTES), "lotesConDatos": TOT["lotes_con_datos"],
                           "pronosticos": TOT["pronosticos"], "eventosNegocio": TOT["eventos_negocio"],
                           "eventosNegocioEnKafka": publicados, "riesgoMax": round(TOT["riesgo_max"], 4),
                           "error": filas_err, "compuertas": compuertas_salida}
assert TOT["pronosticos"] > 0, "El stream no entrego ninguna fila: esta corriendo el productor?"
assert all(compuertas_salida.values()), "Compuertas de salida que fallan: " + str([n for n, ok in compuertas_salida.items() if not ok])

     variable  RMSE_validacion_12b  RMSE_validacion_persistencia  RMSE_vivo_modelo  MAE_vivo_modelo  RMSE_vivo_persistencia  MAE_vivo_persistencia  comparaciones
temperatura_c               0.1299                        0.2365            0.1131           0.0883                  0.1926                 0.1342            363
  oxigeno_mgl               0.1422                        0.2096            0.1260           0.0952                  0.1957                 0.1324            364
OK    El stream entrego filas
OK    Los 4 eventos de negocio emitidos estan en Kafka


## 4. KPI de tiempo: percentiles, no solo el promedio

| KPI | Qué mide | Compuerta |
|---|---|---|
| Latencia de punta a punta | Desde que el puente deja el evento en Kafka hasta que Spark lo pronostica | **p50 < 60 s** (disparador de 10 s más MQTT y puente). El p95 sobre 5 min solo **advierte**: indica lag acumulado |
| Duración del micro-lote | Lo que tarda Spark en procesar un lote | p50 menor que el disparador (10 s) |
| Retraso de publicación del USGS | Medición → publicación, solo eventos en vivo | Informativo (no depende de nosotros) |

La compuerta va sobre el p50, no sobre el promedio: si hay una cola larga (por ejemplo, mensajes que esperaron en Kafka al arrancar), el promedio no describe ni al evento típico ni a los lentos.

In [7]:
def pct(xs, q):
    return float(pd.Series(xs).quantile(q)) if len(xs) else None

lat = {"n": len(LAT_E2E_S), "promedioS": float(pd.Series(LAT_E2E_S).mean()) if LAT_E2E_S else None,
       "p50S": pct(LAT_E2E_S, .5), "p95S": pct(LAT_E2E_S, .95), "maxS": max(LAT_E2E_S) if LAT_E2E_S else None,
       "fraccionMayor1Min": float((pd.Series(LAT_E2E_S) > 60).mean()) if LAT_E2E_S else None}
dur = {"n": len(DURACION_LOTE_MS), "promedioMs": float(pd.Series(DURACION_LOTE_MS).mean()) if DURACION_LOTE_MS else None,
       "p50Ms": pct(DURACION_LOTE_MS, .5), "p95Ms": pct(DURACION_LOTE_MS, .95)}
usgs = {"n": len(RETRASO_USGS_MIN), "p50Min": pct(RETRASO_USGS_MIN, .5), "p95Min": pct(RETRASO_USGS_MIN, .95)}

print(f"Latencia de punta a punta ({lat['n']} eventos): promedio {lat['promedioS']:.1f} s | p50 {lat['p50S']:.1f} s | "
      f"p95 {lat['p95S']:.1f} s | max {lat['maxS']:.1f} s | con mas de 1 min: {lat['fraccionMayor1Min']:.1%}")
print(f"Duracion del micro-lote ({dur['n']} lotes): promedio {dur['promedioMs']:.0f} ms | p50 {dur['p50Ms']:.0f} ms | p95 {dur['p95Ms']:.0f} ms")
if usgs["n"]:
    print(f"Retraso de publicacion USGS ({usgs['n']} eventos en vivo): p50 {usgs['p50Min']:.1f} min | p95 {usgs['p95Min']:.1f} min")
else:
    print("Retraso de publicacion USGS: sin eventos en vivo en esta ventana (el productor reproduce un episodio)")

compuertas_tiempo = {f"Latencia tipica (p50) menor a {P50_LATENCIA_MAX_S} s": lat["p50S"] < P50_LATENCIA_MAX_S,
                     f"Micro-lote tipico (p50) menor al disparador ({TRIGGER_S} s)": dur["p50Ms"] < TRIGGER_S * 1000}
for nombre, ok in compuertas_tiempo.items():
    print(("OK    " if ok else "FALLA ") + nombre)
if lat["p95S"] > P95_LATENCIA_AVISO_S:
    manifiesto["advertencias"].append(f"p95 de latencia {lat['p95S']:.0f} s > {P95_LATENCIA_AVISO_S} s: el promedio no representa al evento tipico")
    print("ADVERTENCIA:", manifiesto["advertencias"][-1])
manifiesto["kpiTiempo"] = {"latenciaPuntaAPunta": lat, "duracionMicrolote": dur, "retrasoUsgs": usgs, "compuertas": compuertas_tiempo}
assert all(compuertas_tiempo.values()), "KPI de tiempo no plausible: " + str([n for n, ok in compuertas_tiempo.items() if not ok])

Latencia de punta a punta (1015 eventos): promedio 6.7 s | p50 6.7 s | p95 11.3 s | max 12.1 s | con mas de 1 min: 0.0%
Duracion del micro-lote (22 lotes): promedio 2123 ms | p50 2034 ms | p95 2652 ms
Retraso de publicacion USGS: sin eventos en vivo en esta ventana (el productor reproduce un episodio)
OK    Latencia tipica (p50) menor a 60 s
OK    Micro-lote tipico (p50) menor al disparador (10 s)


## 5. Publicar en Pushgateway y confirmar en Prometheus

Se compara cada valor que devuelve Prometheus con el **recién publicado**, no solo que exista: si la métrica ya estaba de una corrida anterior, "hay un valor" pasaría sin que la publicación nueva haya llegado. La celda espera hasta 90 s porque Prometheus recoge el dato en su siguiente *scrape* (cada 15 s).

In [8]:
def fnum(x): return 0.0 if x is None else float(x)
metricas = {
    "aquavigia_int_batch_filas": total,
    "aquavigia_int_compuertas_ok": sum(compuertas.values()) + sum(compuertas_calidad.values()) + sum(compuertas_salida.values()) + sum(compuertas_tiempo.values()),
    "aquavigia_int_stream_pronosticos": TOT["pronosticos"],
    "aquavigia_int_eventos_negocio": TOT["eventos_negocio"],
    "aquavigia_int_riesgo_max": round(TOT["riesgo_max"], 4),
    "aquavigia_int_latencia_p50_s": round(fnum(lat["p50S"]), 3),
    "aquavigia_int_latencia_p95_s": round(fnum(lat["p95S"]), 3),
    "aquavigia_int_latencia_promedio_s": round(fnum(lat["promedioS"]), 3),
    "aquavigia_int_lote_p50_ms": round(fnum(dur["p50Ms"]), 1),
    "aquavigia_int_lote_p95_ms": round(fnum(dur["p95Ms"]), 1),
    "aquavigia_int_ultima_ejecucion_timestamp": int(time.time()),
}
if usgs["n"]:
    metricas["aquavigia_int_retraso_usgs_p50_min"] = round(usgs["p50Min"], 3)
    metricas["aquavigia_int_retraso_usgs_p95_min"] = round(usgs["p95Min"], 3)
for r in filas_err:
    for m in ("modelo", "persistencia"):
        if r[f"RMSE_vivo_{m}"] is not None:
            metricas[f'aquavigia_int_rmse_vivo{{variable="{r["variable"]}",modelo="{m}"}}'] = r[f"RMSE_vivo_{m}"]

def nombre_base(n): return n.split("{")[0]
tipos = sorted({nombre_base(n) for n in metricas})
cuerpo = "".join(f"# TYPE {t} gauge\n" for t in tipos) + "".join(f"{n} {v}\n" for n, v in metricas.items())
url = f"{PUSHGATEWAY}/metrics/job/u3_integracion/instance/{ID}"
with urllib.request.urlopen(urllib.request.Request(url, data=cuerpo.encode(), method="PUT"), timeout=10) as r:
    print("Pushgateway respondio:", r.status)

def consultar(n):
    if "{" in n:
        base, lab = n.split("{", 1)
        consulta = f'{base}{{job="u3_integracion",instance="{ID}",{lab[:-1]}}}'
    else:
        consulta = f'{n}{{job="u3_integracion",instance="{ID}"}}'
    res = json.load(urllib.request.urlopen(PROMETHEUS + "/api/v1/query?query=" + urllib.parse.quote(consulta), timeout=10))["data"]["result"]
    return res[0]["value"][1] if res else None

def coincide(obtenido, esperado):
    return obtenido is not None and abs(float(obtenido) - float(esperado)) <= 1e-6 * max(1.0, abs(float(esperado)))

confirmadas, limite = {}, time.time() + 90
while time.time() < limite:
    confirmadas = {n: consultar(n) for n in metricas}
    if all(coincide(confirmadas[n], v) for n, v in metricas.items()): break
    time.sleep(5)
for n, v in metricas.items():
    print(f"{'OK    ' if coincide(confirmadas[n], v) else 'FALLA '}{n}: Prometheus={confirmadas[n]}  publicado={v}")
todas = all(coincide(confirmadas[n], v) for n, v in metricas.items())
manifiesto["observabilidad"] = {"metricas": {n: str(v) for n, v in metricas.items()}, "todasEnPrometheus": todas}
assert todas, "Prometheus no devolvio los valores recien publicados"

Pushgateway respondio: 200
OK    aquavigia_int_batch_filas: Prometheus=1245817  publicado=1245817
OK    aquavigia_int_compuertas_ok: Prometheus=21  publicado=21
OK    aquavigia_int_stream_pronosticos: Prometheus=1015  publicado=1015
OK    aquavigia_int_eventos_negocio: Prometheus=4  publicado=4
OK    aquavigia_int_riesgo_max: Prometheus=0.7447  publicado=0.7447
OK    aquavigia_int_latencia_p50_s: Prometheus=6.715  publicado=6.715
OK    aquavigia_int_latencia_p95_s: Prometheus=11.275  publicado=11.275
OK    aquavigia_int_latencia_promedio_s: Prometheus=6.728  publicado=6.728
OK    aquavigia_int_lote_p50_ms: Prometheus=2033.5  publicado=2033.5
OK    aquavigia_int_lote_p95_ms: Prometheus=2652.2  publicado=2652.2
OK    aquavigia_int_ultima_ejecucion_timestamp: Prometheus=1794587731  publicado=1794587731
OK    aquavigia_int_rmse_vivo{variable="temperatura_c",modelo="modelo"}: Prometheus=0.1131  publicado=0.1131
OK    aquavigia_int_rmse_vivo{variable="temperatura_c",modelo="persistencia"}: P

## 6. Manifiesto de la ejecución y comparación con la corrida anterior

Un pipeline con fuente en vivo no es reproducible valor por valor: dos corridas procesan lecturas distintas. Lo reproducible es el **procedimiento** (mismas compuertas y dependencias); lo auditable es **qué dio cada corrida**. El manifiesto queda en `artifacts/manifiestos/` (no se versiona; se guarda para la sustentación).

In [9]:
manifiesto["duracionTotalS"] = round(time.time() - INICIO, 1)
manifiesto["finUtc"] = datetime.now(timezone.utc).isoformat()
ruta = f"{DIR_MANIFIESTOS}/manifiesto_{ID}_{RUN_ID}.json"
with open(ruta, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)
print("Manifiesto escrito en", ruta, "| duracion total:", manifiesto["duracionTotalS"], "s")

def resumen(m):
    s = m["streaming"]; t = m["kpiTiempo"]; e = {r["variable"]: r for r in s["error"]}
    fila = {"runId": m["runId"], "duracion_s": m["duracionTotalS"], "filas_batch": m["batch"]["filas"],
            "pronosticos": s["pronosticos"], "eventos_negocio": s["eventosNegocio"], "riesgo_max": s["riesgoMax"],
            "lat_p50_s": round(t["latenciaPuntaAPunta"]["p50S"], 1), "lat_p95_s": round(t["latenciaPuntaAPunta"]["p95S"], 1),
            "lote_p50_ms": round(t["duracionMicrolote"]["p50Ms"]), "advertencias": len(m["advertencias"])}
    for v in VARIABLES:
        fila[f"rmse_vivo_{v}"] = e[v]["RMSE_vivo_modelo"]
    return fila

previos = sorted(p for p in os.listdir(DIR_MANIFIESTOS) if p.startswith(f"manifiesto_{ID}_"))
tabla = pd.DataFrame([resumen(json.load(open(f"{DIR_MANIFIESTOS}/{p}", encoding="utf-8"))) for p in previos[-2:]])
print(f"\nComparacion de las dos ultimas corridas ({len(previos)} manifiestos en total):")
print(tabla.T.to_string(header=False))
print("\nEsperable entre corridas: cambian pronosticos, eventos, latencias y RMSE en vivo (lecturas distintas).")
print("Regresion: una compuerta que deja de pasar, o el stream con cero filas.")

Manifiesto escrito en /opt/AquaVigia_Producto_U3/patricia/artifacts/manifiestos/manifiesto_patricia_20261113T163113Z.json | duracion total: 263.0 s

Comparacion de las dos ultimas corridas (3 manifiestos en total):
runId                    20261113T104634Z  20261113T163113Z
duracion_s                          254.8             263.0
filas_batch                       1245817           1245817
pronosticos                           971              1015
eventos_negocio                         4                 4
riesgo_max                         0.7447            0.7447
lat_p50_s                             7.4               6.7
lat_p95_s                            11.9              11.3
lote_p50_ms                          2883              2034
advertencias                            0                 0
rmse_vivo_temperatura_c            0.1127            0.1131
rmse_vivo_oxigeno_mgl               0.121             0.126

Esperable entre corridas: cambian pronosticos, eventos, latencia